# A brick wall is a graph: sequence, load path, what falls

Companion to [Week 06 §8](../README.md#8-graphs-of-building-elements).
The notebook before this one put two bricks, then six, in a graph.
Here it is a wall with a window: about a hundred bricks, where the graph
starts to answer questions you cannot answer by looking.

It finds the contact between two bricks with a mesh boolean.
For a hundred bricks that is thousands of booleans and minutes of waiting.
In a coursed wall an interval overlap gives the same graph instantly.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in Lecture/notebooks/ or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from compas.datastructures import Graph

## Step 1 — the wall as data
Running bond: every second course is shifted by half a brick. A window is
cut out of courses 3 to 7 and a concrete lintel spans it in course 8.
A brick is three numbers: where it starts, where it ends, which course.

In [ ]:
MODULE = 0.24                    # one brick (0.23 m) plus one joint (0.01 m)
COURSE = 0.06                    # one brick height (0.05 m) plus one bed joint
LENGTH, COURSES = 2.40, 12
WINDOW = (0.72, 1.68, 3, 8)      # x from, x to, first course, course of the lintel
BEARING = 0.24                   # how far the lintel sits on the wall each side


def cut(x0, x1, void):
    """The parts of the interval (x0, x1) that are outside the void."""
    v0, v1 = void
    parts = [(x0, min(x1, v0)), (max(x0, v1), x1)]
    return [(a, b) for a, b in parts if b - a > 0.05]


def build_wall(lintel=True):
    wx0, wx1, first, top = WINDOW
    pieces = []                                              # (x0, x1, course, kind)
    for course in range(COURSES):
        shift = MODULE / 2 if course % 2 else 0.0
        void = (wx0, wx1) if first <= course < top else (wx0 - BEARING, wx1 + BEARING) if course == top else None
        x = -shift
        while x < LENGTH:
            x0, x1 = max(x, 0.0), min(x + MODULE - 0.01, LENGTH)
            for a, b in ([(x0, x1)] if void is None else cut(x0, x1, void)):
                pieces.append((a, b, course, "brick"))
            x += MODULE
        if course == top and lintel:
            pieces.append((wx0 - BEARING, wx1 + BEARING, course, "lintel"))
    return sorted(pieces, key=lambda piece: (piece[2], piece[0]))


def rectangle(piece):
    x0, x1, course, _ = piece
    z = course * COURSE
    return [(x0, z), (x1, z), (x1, z + COURSE - 0.01), (x0, z + COURSE - 0.01)]


def draw_wall(pieces, title, fills=None, lines=(), labels=False, size=(900, 330)):
    centres = [((p[0] + p[1]) / 2, p[2] * COURSE + 0.025) for p in pieces]
    return draw_points(centres if labels else [], title, polygons=[rectangle(p) for p in pieces],
                       polygon_fills=fills or ["#ffd8a8" if p[3] == "brick" else "#ced4da" for p in pieces],
                       lines=lines, labels=labels, size=size, bounds=[(0, 0), (LENGTH, COURSES * COURSE)])


pieces = build_wall()
print(len(pieces), "pieces:", sum(p[3] == "brick" for p in pieces), "bricks and 1 lintel")
draw_wall(pieces, "the wall (grey = lintel); the numbers are node keys", labels=True)

## Step 2 — "rests on" is an edge
A piece rests on a piece in the course below when their x-intervals overlap.
Add the edge **from the upper piece to the lower one**: COMPAS keeps the
direction you give (Week 06 §2), and here the direction means something.

- `neighbors_out(n)`: what `n` rests on
- `neighbors_in(n)`: what rests on `n`

In [ ]:
def wall_graph(pieces):
    graph = Graph()
    for key, (x0, x1, course, kind) in enumerate(pieces):
        graph.add_node(key, x=(x0 + x1) / 2, y=course * COURSE + 0.025, z=0.0,
                       x0=x0, x1=x1, course=course, kind=kind)
    for upper, (a0, a1, course, _) in enumerate(pieces):
        for lower, (b0, b1, below, _) in enumerate(pieces):
            overlap = min(a1, b1) - max(a0, b0)
            if below == course - 1 and overlap > 0.02:
                graph.add_edge(upper, lower, overlap=overlap)
    return graph


wall = wall_graph(pieces)
print(wall.number_of_nodes(), "nodes,", wall.number_of_edges(), "edges")
n = 40
print(f"piece {n} rests on {list(wall.neighbors_out(n))} and carries {list(wall.neighbors_in(n))}")
draw_wall(pieces, "every edge is one piece resting on another",
          lines=[(wall.node_coordinates(u), wall.node_coordinates(v)) for u, v in wall.edges()])

### How well is each piece supported?
The out-degree counts supports. Two is the bond working as designed. One
means the piece sits on a single brick. Zero should only happen in course 0.

In [ ]:
supports = {n: len(list(wall.neighbors_out(n))) for n in wall.nodes()}
for count in sorted(set(supports.values())):
    print(f"{count} supports: {sum(s == count for s in supports.values()):3d} pieces")
floating = [n for n in wall.nodes() if supports[n] == 0 and wall.node_attribute(n, "course") > 0]
print("floating pieces:", floating)
shade = {0: "#ced4da", 1: "#ffa94d", 2: "#d0ebff"}
draw_wall(pieces, "orange = resting on one piece only",
          fills=[shade.get(supports[n], "#74c0fc") for n in wall.nodes()])

## Step 3 — follow the supports down
Pick a brick near the top and walk down along `neighbors_out` until a brick
in course 0. **Depth-first search**: always take the first support, and
only come back to try another one if the first leads nowhere.

In [ ]:
def path_to_ground(graph, start):
    """Depth-first: one chain of supports from start down to course 0, or None."""
    if graph.node_attribute(start, "course") == 0:
        return [start]
    for below in graph.neighbors_out(start):
        rest = path_to_ground(graph, below)
        if rest:
            return [start] + rest
    return None


def everything_under(graph, start):
    """Every piece that start rests on, directly or not."""
    seen, stack = set(), [start]
    while stack:
        for below in graph.neighbors_out(stack.pop()):
            if below not in seen:
                seen.add(below)
                stack.append(below)
    return seen


top_brick = max(wall.nodes(), key=lambda n: (wall.node_attribute(n, "course"), -abs(wall.node_attribute(n, "x") - 1.2)))
chain = path_to_ground(wall, top_brick)
cone = everything_under(wall, top_brick)
print(f"from brick {top_brick}: one chain to the ground is", chain)
print(f"it rests, directly or not, on {len(cone)} of the {wall.number_of_nodes() - 1} other pieces")
draw_wall(pieces, "orange: one chain of supports; blue: everything this brick rests on",
          fills=["#ff922b" if n in chain else "#a5d8ff" if n in cone else "#f1f3f5" for n in wall.nodes()])

The chain is one way down; the blue region is the brick's **load cone**:
every piece that has to be there for this brick to stay up. It widens by
half a brick per course, and it steps around the window through the lintel.
No coordinate was compared to find it: only edges were followed.

## Step 4 — a building sequence, from the graph alone
A piece can be laid once everything it rests on is laid. Repeat: take a
piece with no unlaid supports, lay it, tell the pieces above it. This is
**Kahn's algorithm**; the order it produces is a *topological order*.

Nothing here uses the course number. The same loop would sequence a timber
frame or a precast assembly, where there are no courses to count.

In [ ]:
def sequence(graph):
    waiting = {n: len(list(graph.neighbors_out(n))) for n in graph.nodes()}
    ready = [n for n in graph.nodes() if waiting[n] == 0]
    order = []
    while ready:
        n = ready.pop(0)
        order.append(n)
        for above in graph.neighbors_in(n):
            waiting[above] -= 1
            if waiting[above] == 0:
                ready.append(above)
    return order


order = sequence(wall)
step = {n: i for i, n in enumerate(order)}
print("first ten:", order[:10])
print("the lintel is step", step[next(wall.nodes_where(kind="lintel"))], "of", len(order))
ok = all(step[u] > step[v] for u, v in wall.edges())
print("every piece is laid after its supports:", ok)
draw_wall(pieces, "building sequence: blue first, orange last",
          fills=[ramp(step[n] / len(order)) for n in wall.nodes()])

## Step 5 — where does the weight go?
Walk the sequence **backwards**, top to bottom. Each piece carries its own
weight plus what arrived from above, and passes the total down to its
supports in proportion to the overlap. One pass, because the order
guarantees a piece is finished before its supports are visited.

In [ ]:
def weight(graph, n):
    x0, x1, kind = graph.node_attributes(n, ["x0", "x1", "kind"])
    density = 2400 if kind == "lintel" else 1800             # kg/m3
    return (x1 - x0) * 0.11 * 0.05 * density


def loads(graph, order):
    carried = {n: weight(graph, n) for n in graph.nodes()}
    for n in reversed(order):
        below = list(graph.neighbors_out(n))
        total = sum(graph.edge_attribute((n, s), "overlap") for s in below)
        for s in below:
            carried[s] += carried[n] * graph.edge_attribute((n, s), "overlap") / total
    return carried


carried = loads(wall, order)
on_ground = sum(carried[n] for n in wall.nodes_where(course=0))
print(f"weight of the wall {sum(weight(wall, n) for n in wall.nodes()):.1f} kg; "
      f"arriving at the ground {on_ground:.1f} kg")
lintel = next(wall.nodes_where(kind="lintel"))
print(f"the lintel carries {carried[lintel]:.1f} kg and hands it to pieces {list(wall.neighbors_out(lintel))}")
heaviest = max(wall.nodes_where(kind="brick"), key=carried.get)
print(f"most loaded brick: {heaviest} with {carried[heaviest]:.1f} kg, "
      f"{carried[heaviest] / weight(wall, heaviest):.0f} times its own weight")
top = carried[heaviest]                  # scale to the bricks; the lintel is off the scale
draw_wall(pieces, "load carried by each piece", fills=[ramp(carried[n] / top) for n in wall.nodes()])

The two totals agree: nothing was lost on the way down, which is the check
that the pass is right. The picture shows what the lintel does: the load
above the window is sent sideways and comes down the two **jambs**.

## Step 6 — take the lintel away
Which pieces lose their support? A piece stands if it is on the ground or
rests on at least one piece that stands. Going through the sequence once
decides it for every piece.

In [ ]:
def standing(graph, order):
    stands = set()
    for n in order:
        below = list(graph.neighbors_out(n))
        if graph.node_attribute(n, "course") == 0 or any(s in stands for s in below):
            stands.add(n)
    return stands


bare = build_wall(lintel=False)
bare_graph = wall_graph(bare)
laid = sequence(bare_graph)
print("sequence() still lays", len(laid), "of", len(bare), "pieces: a piece with no supports looks like ground to it")
fallen = [n for n in bare_graph.nodes() if n not in standing(bare_graph, laid)]
print("without support:", fallen)
draw_wall(bare, "no lintel: red pieces have nothing under them",
          fills=["#ff8787" if n in fallen else "#ffd8a8" for n in bare_graph.nodes()])

Two things to take from this.

The sequence did not notice anything wrong. Kahn's algorithm lays a brick in
mid-air without complaint, because "no supports" is also what the ground
course looks like. An algorithm answers the question you asked; the
*floating* check in Step 2 is the one that asks the right question.

The red pieces form a stepped triangle (cut off here by the top of the
wall): each course reaches half a brick further over the opening than the
one below. That is a **corbel**, the oldest way to span an opening, and the
reason a lintel carries a triangle of wall and not everything above it.

Be clear about what this is: **connectivity, not statics.** A brick hanging
on a 20 mm overlap "stands" here. The graph says what *could* hold; whether
it does is a question for the truss notebook's kind of analysis.

## Step 7 — the wall is a file
Nodes, edges and every attribute on them go into one JSON text and come
back unchanged, direction of the edges included. This is the artifact you
hand to the next tool (a viewer, a robot program, Rhino): the graph is
computed once, here, and only *read* there.

In [ ]:
import compas

text = compas.json_dumps(wall)
again = compas.json_loads(text)
print(f"{len(text) / 1000:.0f} kB of JSON")
print("same nodes and edges:", again.number_of_nodes() == wall.number_of_nodes(),
      again.number_of_edges() == wall.number_of_edges())
print("attributes survive:  ", again.node_attributes(lintel))
print("direction survives:  ", sorted(again.neighbors_out(lintel)) == sorted(wall.neighbors_out(lintel)))
print("the sequence from the reloaded graph is the same:", sequence(again) == order)

`compas.json_dump(wall, "wall.json")` writes the same text to a file, and
`compas.json_load` reads it back.

### The wall in 3D, coloured by sequence

In [ ]:
from compas.colors import Color
from compas.datastructures import Mesh

solid, colours = Mesh(), {}
sides = [(0, 3, 2, 1), (4, 5, 6, 7), (0, 1, 5, 4), (1, 2, 6, 5), (2, 3, 7, 6), (3, 0, 4, 7)]
for n, (x0, x1, course, kind) in enumerate(pieces):
    z0, z1 = course * COURSE, course * COURSE + 0.05
    keys = [solid.add_vertex(x=x, y=y, z=z) for z in (z0, z1) for x, y in ((x0, 0), (x1, 0), (x1, 0.11), (x0, 0.11))]
    for side in sides:
        colours[solid.add_face([keys[i] for i in side])] = Color.from_hex(ramp(step[n] / len(order)))
view3d((solid, {"facecolor": colours, "show_edges": True}))

## Your turn
1. `ready.pop(0)` takes the oldest ready piece. Change it to `ready.pop()`.
   The order is still valid: what kind of mason builds this way?
2. Widen the window by two bricks. What happens to the load on the jambs,
   and to the size of the triangle in Step 6?
3. Bricks also touch their neighbours in the same course. Add those *head*
   joints as edges with `kind="head"`, and tag the existing ones
   `kind="bed"`. Which steps must now filter by `kind`, and why?
4. Set `BEARING = 0.0` so the lintel only just fills the opening. What does
   Step 2's support count say about the lintel now?